In [1]:
import numpy as np
import pandas as pd

from exerpy import ExergyAnalysis
from exerpy.analyses import _load_json

In [2]:
def diff_between_simulators(testcase):
    simulator_results = []
    for path in testcase.values():
        contents = _load_json(path)
        if "settings" not in contents:
            contents["settings"] = {}

        simulator_results += [
            ExergyAnalysis.from_json(path, **contents["settings"])
        ]

    sim1 = simulator_results[0]
    sim2 = simulator_results[1]

    columns = ["m", "p", "T"]
    if sim1.chemExLib is not None:
        columns.append("e_CH")
    if sim1.split_physical_exergy:
        columns.append("e_M")
        columns.append("e_T")
    else:
        columns.append("e_PH")

    df_sim1 = pd.DataFrame.from_dict(
        sim1._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")
    df_sim2 = pd.DataFrame.from_dict(
        sim2._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")

    overlapping_index = list(
        set(df_sim1.index.tolist()) & set(df_sim2.index.tolist())
    )
    df_sim1 = df_sim1.loc[overlapping_index].round(6)
    df_sim2 = df_sim2.loc[overlapping_index].round(6)

    # inf means that sim2 has 0 value, comparison does not make sense there
    # and sometimes there seem to be NaN values in the dataframes, those are
    # removed as well
    return (
        (df_sim1 - df_sim2) / df_sim2
    ).abs().replace(np.inf, 0).fillna(0)

In [3]:
testcase = {
    "ebsilon": "cgam_ebs.json",
    "tespy": "cgam_tespy.json"
}
diff_between_simulators(testcase)


,m,p,T,e_CH,e_PH
3,0.000183,0.000052,0.000000e+00,0.000004,5.555921e-04
1,0.000183,0.000000,0.000000e+00,0.000004,0.000000e+00
10,0.001399,0.000000,0.000000e+00,0.000000,7.728223e-03
2,0.000183,0.000000,8.233846e-04,0.000004,1.452651e-03
8,0.000000,0.000000,0.000000e+00,0.000000,2.469128e-08
8P,0.000000,0.000000,0.000000e+00,0.000000,3.226984e-10
4,0.000155,0.000036,1.973684e-09,0.003343,6.022982e-04
5,0.000155,0.000268,3.329500e-04,0.003343,3.759011e-04
6P,0.000155,0.000177,8.884515e-04,0.003343,2.640128e-03
7,0.000155,0.000000,1.002384e-03,0.003343,4.907438e-03


In [4]:
testcase = {
    "tespy": "cgam_tespy.json",
    "aspen": "cgam_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_CH,e_PH
3,0.000128,0.000052,0.000000e+00,0.000000,9.748562e-04
1,0.000128,0.000000,0.000000e+00,0.000000,0.000000e+00
10,0.000724,0.000000,0.000000e+00,0.000000,7.640852e-03
2,0.000128,0.000000,2.891440e-04,0.000000,1.153838e-03
8,0.000000,0.000000,0.000000e+00,0.000000,6.382963e-07
8P,0.000000,0.000000,4.781872e-07,0.000000,3.005818e-06
4,0.000113,0.000036,6.578948e-10,0.001816,6.418568e-04
5,0.000093,0.000268,9.613326e-05,0.001769,6.717224e-04
6P,0.000093,0.000177,3.489563e-04,0.001769,1.336945e-03
7,0.000093,0.000000,5.023293e-04,0.001769,2.875732e-03


In [5]:
testcase = {
    "ebsilon": "cgam_ebs.json",
    "aspen": "cgam_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_CH,e_PH
3,0.000055,0.000000e+00,0.000000e+00,0.000004,4.187225e-04
1,0.000055,0.000000e+00,0.000000e+00,0.000004,0.000000e+00
10,0.000675,0.000000e+00,0.000000e+00,0.000000,2.832130e-05
2,0.000055,0.000000e+00,5.344786e-04,0.000004,3.004893e-04
8,0.000000,0.000000e+00,0.000000e+00,0.000000,6.629876e-07
8P,0.000000,0.000000e+00,4.781872e-07,0.000000,3.005495e-06
4,0.000042,0.000000e+00,1.315789e-09,0.001533,3.917200e-05
5,0.000061,4.094631e-10,2.368488e-04,0.001580,2.955688e-04
6P,0.000061,4.329004e-10,5.398052e-04,0.001580,1.306713e-03
7,0.000061,0.000000e+00,5.005582e-04,0.001580,2.045818e-03


In [6]:
# Component results stored by the example scripts cgam_ebs.py, cgam_tespy.py and cgam_aspen.py
results = {
    simulator: pd.read_csv(f"cgam_components_{simulator.lower()}.csv", index_col=0).set_index("Component")
    for simulator in ("Ebsilon", "TESPy", "Aspen")
}

# Components the simulators model differently are compared as one group
groups = {
    "COMB": "CC",  # Aspen name of the combustion chamber
    "DRUM": "EV",  # TESPy models the drum separately
}

# Exergy destruction in kW of the components found in at least two models, and the deviations from Ebsilon
E_D = (
    pd.DataFrame({simulator: df["E_D [kW]"] for simulator, df in results.items()})
    .drop("TOT")
    .rename(index=groups)
    .groupby(level=0)
    .sum(min_count=1)
    .dropna(thresh=2)
)
reference = E_D["Ebsilon"].where(E_D["Ebsilon"].abs() > 1e-3)  # no relative deviation below 1 W
for simulator in ("TESPy", "Aspen"):
    E_D[f"{simulator} vs Ebsilon [%]"] = (E_D[simulator] / reference - 1) * 100
E_D.sort_values("Ebsilon", ascending=False).round(2)

,Ebsilon,TESPy,Aspen,TESPy vs Ebsilon [%],Aspen vs Ebsilon [%]
Component,,,,,
CC,25314.35,25397.79,25359.89,0.33,0.18
EV,4586.83,4600.59,4593.98,0.30,0.16
EXP,2994.12,2995.01,2993.39,0.03,-0.02
APH,2559.56,2557.50,2556.98,-0.08,-0.10
AC,2100.49,2101.22,2099.82,0.03,-0.03
PH,1911.39,1915.95,1914.67,0.24,0.17
GEN,0.00,NaN,0.00,NaN,NaN


In [7]:
# Results of the whole plant
columns = ["E_F [kW]", "E_P [kW]", "E_D [kW]", "E_L [kW]", "epsilon [%]"]
pd.DataFrame({simulator: df.loc["TOT", columns] for simulator, df in results.items()}).round(3)

,Ebsilon,TESPy,Aspen
E_F [kW],85081.016,85195.392,85138.501
E_P [kW],42753.645,42753.646,42753.638
E_D [kW],39466.737,39568.062,39518.722
E_L [kW],2860.633,2873.683,2866.140
epsilon [%],50.251,50.183,50.217
